<a href="https://colab.research.google.com/drive/1ZO8ULhTfWUUTIYNO2Kml5Rw1RdM8g5OR?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

matplotlib version: 3.10.1
torch version: 2.5.1
tiktoken version: 0.9.0


**Coding an LLM architecture**

In [6]:
import torch
import torch.nn as nn

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": True       # Query-Key-Value bias
}

**Normalizing activations with layer normalization**

In [7]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

**Connecting attention and linear layers in a transformer block**

In [8]:
from attention import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

--- TEST 1 (d_out=2, heads=2, temp=1.0) ---
Output shape: torch.Size([2, 6, 2])
First token output ('Your'):
 [0.319  0.4858]

--- TEST 2 (d_out=4, heads=2, temp=0.5) ---
Output shape: torch.Size([2, 6, 4])
First token output ('Your'):
 [ 0.6241 -0.5129 -0.4382 -0.0106]

--- TEST 3 (d_out=4, heads=2, temp=2.0) ---
Output shape: torch.Size([2, 6, 4])
First token output ('Your'):
 [ 0.1103 -0.1281 -0.1156 -0.3116]


In [9]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [10]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

**Text-completion Function** (with loop) (Greedy Decoding - immer das wahrscheinlichste Token wird ausgewählt)

In [11]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

**Test on untrained model**

In [12]:
import torch
import tiktoken  # Falls nicht installiert: in neuer Zelle '!pip install tiktoken' ausführen

# 1. Tokenizer laden (GPT-2 BPE-Encoding)
tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
# 2. Text definieren & enkodieren
start_context = "Hello, my name is"
encoded = tokenizer.encode(start_context)
#print("encoded:", encoded)

# 3. In PyTorch-Tensor umwandeln & Batch-Dimension hinzufügen
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
#print("encoded_tensor.shape:", encoded_tensor.shape)

model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

#print("Output:", out)
#print("Output length:", len(out[0]))

# untrainiertes Modell
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print("\n")
print(decoded_text)



Hello, my name is Empires Conticts exacerbate adjunctrast agony Reservcontroller hospitals


**Test on GPT-2**

In [ ]:
from huggingface_hub import hf_hub_download

# Vortrainierte Gewichte herunterladen
checkpoint = hf_hub_download(
    repo_id="rasbt/gpt2-from-scratch-pytorch",
    filename="gpt2-small-124M.pth"
)

# Gewichte in das Modell laden
state_dict = torch.load(checkpoint, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()

# Erneuter Test – mit GPT-2-Gewichten
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Trainierte Ausgabe:", tokenizer.decode(out.squeeze(0).tolist()))

c:\Users\rebec\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Test: Text generation with Temperature Scaling und Top-k Sampling (for more variance)**

In [ ]:
import torch

def generate_text_with_sampling(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # 1. Top-k Filtering (nur die k wahrscheinlichsten Tokens berücksichtigen)
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # 2. Temperature Scaling & Stochastisches Sampling
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            # Fallback zu Greedy Decoding bei temperature = 0.0
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
torch.manual_seed(123)

start_context = "Hello, my name is"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

model.eval()

# Generierung mit Sampling
out = generate_text_with_sampling(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=100,
    context_size=GPT_CONFIG_124M["context_length"],
    temperature=0.8,  # Bringt Vielfalt/Varianz -> höhrere Temperatur: mehr Varianz & Kreativität, niedrigere Temperatur -> Text wird sehr fokussiert, präzise und vorhersehbar.
    top_k=30          # Verhindert unsinnige Tokens, beschränkt die Tokens auf die k-wahrscheinlichsten Token
)

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print("Ausgabe mit Sampling:\n", decoded_text)

Ausgabe mit Sampling:
 Hello, my name is Kiyoshi Shimizu. You may remember me from my time as a young person, I was very close to the people who were trying to save us so that we could get to the next generation. The people who are trying to save us now have very good reasons to be worried.

Yoshitaka:

When I was younger, many people thought that I was a fraud. I started out with a plan in mind. I needed to save the world, to get the
